# Clustering Jerarquico — Caso completo: segmentacion bancaria

---

**Autor:** Borja Mora Méndez
**Contacto:** [borja.mora.mendez@gmail.com](mailto:borja.mora.mendez@gmail.com) · [LinkedIn](https://www.linkedin.com/in/borja-mora-mendez/)
**Repositorio:** [Data Analytics Portfolio](https://github.com/BORJAMOME/Data-Analytics-Portfolio)
**Categoría:** Machine Learning · No Supervisado · Clustering · Jerárquico

---

**Objetivo:** Pipeline completo de clustering jerarquico sobre un dataset sintetico de clientes bancarios con 5 variables. Se comparan metricas de distancia, metodos de enlace, coeficiente cofenetico y se valida con silhouette score.

**Contexto de negocio:** Un banco quiere segmentar su base de 300 clientes para asignar gestores especializados, disenar productos financieros por segmento y priorizar campanas de cross-selling.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster, cophenet, set_link_color_palette
from scipy.spatial.distance import pdist
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, silhouette_samples

sns.set_style("whitegrid")
np.random.seed(42)

# Estilo visual — sistema de color validado (consejo UX/UI Data)
BACKGROUND    = '#fbfbfb'
PURPLE        = '#7a7bff'   # único color de énfasis / serie única en scatter y líneas (1 por gráfico)
PURPLE_LIGHT  = '#9b9cff'   # EDA de una sola serie (histogramas independientes)
POSITIVE      = '#6b8158'   # exclusivo signo positivo
NEGATIVE      = '#c34031'   # exclusivo signo negativo
NEUTRAL_BAR   = '#d9d9d9'   # barras/áreas de contexto (siempre con etiqueta de valor)
NEUTRAL_LINE  = '#8f8c9e'   # líneas de contexto
CONTEXT_LINES = [NEUTRAL_LINE, '#a89a8a', '#7d94a8']   # gama fija para 2+ líneas de contexto
INK           = '#111111'
MUTED         = '#707070'

# Paleta categórica para identidad de cluster — validada (ΔE OKLab, simulación CVD) para
# pares adyacentes (barras, líneas, enlaces de dendrograma). En scatter/PCA/t-SNE con 4+ clusters
# el color por sí solo no basta para daltonismo severo: por eso cada cluster lleva también
# una forma de marcador distinta (CLUSTER_MARKERS) — nunca dependas solo del color.
CLUSTER_PALETTE = ['#7a7bff', '#eb6834', '#1baf7a', '#e34948', '#eda100', '#e87ba4', '#008300']
CLUSTER_MARKERS = ['o', 's', '^', 'D', 'v', 'P', 'X']

DIVERGING_CMAP = LinearSegmentedColormap.from_list(
    "borja_diverging", ["#c34031", "#e0a89f", "#f0ede8", "#b7c2a9", "#6b8158"]
)
SEQUENTIAL_GREEN = LinearSegmentedColormap.from_list(
    "borja_sequential", [BACKGROUND, POSITIVE]
)

def color_annotations(ax, values, threshold, dark="#ffffff", light=INK):
    """Recolorea el texto de un heatmap celda a celda según su magnitud."""
    for text, value in zip(ax.texts, np.asarray(values).flatten()):
        text.set_color(dark if abs(value) >= threshold else light)

plt.rcParams.update({
    'figure.figsize': (10, 5),
    'figure.dpi': 100,
    'figure.facecolor': BACKGROUND,
    'axes.facecolor': BACKGROUND,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.edgecolor': MUTED,
    'axes.labelcolor': INK,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'axes.titlecolor': INK,
    'xtick.color': MUTED,
    'ytick.color': MUTED,
    'font.family': 'sans-serif',
    'font.size': 10,
    'grid.color': '#f0f0f0',
    'grid.linewidth': 0.5,
})

print("Librerias cargadas correctamente")

## 1. Generacion del dataset sintetico

Simulamos 300 clientes bancarios con 5 variables: saldo medio, transacciones mensuales, antiguedad, productos contratados e ingresos anuales. Creamos 4 segmentos naturales.

In [2]:
from sklearn.datasets import make_blobs

X_raw, y_true = make_blobs(
    n_samples=[80, 70, 90, 60],
    n_features=5,
    centers=[
        [5000, 15, 24, 2, 30000],    # Basico
        [25000, 40, 60, 4, 55000],   # Medio
        [80000, 80, 120, 6, 95000],  # Premium
        [150000, 20, 180, 8, 150000] # Patrimonial
    ],
    cluster_std=[3000, 8000, 15000, 20000],
    random_state=42
)

columnas = ["Saldo_Medio", "Transacciones_Mes", "Antiguedad_Meses", "Productos", "Ingresos_Anuales"]
df = pd.DataFrame(np.abs(X_raw), columns=columnas).round(0)
df["Productos"] = df["Productos"].clip(1, 10).astype(int)

print(f"Dataset: {df.shape[0]} clientes, {df.shape[1]} variables")
display(df.describe().round(0))

Dataset: 300 clientes, 5 variables


,Saldo_Medio,Transacciones_Mes,Antiguedad_Meses,Productos,Ingresos_Anuales
count,300.0,300.0,300.0,300.0,300.0
mean,61296.0,9918.0,8400.0,10.0,81534.0
std,54306.0,10504.0,8945.0,0.0,46617.0
min,348.0,41.0,2.0,10.0,22141.0
25%,8868.0,2366.0,2083.0,10.0,34899.0
50%,41998.0,6183.0,5543.0,10.0,72322.0
75%,95874.0,13734.0,11111.0,10.0,111783.0
max,190122.0,51184.0,43326.0,10.0,194210.0


## 2. Exploracion visual

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.ravel()

for i, col in enumerate(columnas):
    axes[i].hist(df[col], bins=25, color=PURPLE_LIGHT, edgecolor="white", alpha=0.8)
    axes[i].set_title(col, fontsize=11)
    axes[i].set_ylabel("Frecuencia")

# Heatmap de correlacion
corr = df.corr()
ax_corr = sns.heatmap(corr, annot=True, fmt=".2f", cmap=DIVERGING_CMAP, vmin=-1, vmax=1, center=0, ax=axes[5],
                       square=True, linewidths=2, linecolor=BACKGROUND)
color_annotations(ax_corr, corr.values, threshold=0.6)
axes[5].set_title("Correlacion", fontsize=11)

plt.suptitle("Exploracion de variables bancarias", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## 3. Estandarizacion

In [4]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)

print("Variables estandarizadas (media=0, std=1):")
for i, col in enumerate(columnas):
    print(f"  {col:25s} — media: {X_scaled[:, i].mean():.4f}, std: {X_scaled[:, i].std():.4f}")

Variables estandarizadas (media=0, std=1):
  Saldo_Medio               — media: 0.0000, std: 1.0000
  Transacciones_Mes         — media: -0.0000, std: 1.0000
  Antiguedad_Meses          — media: 0.0000, std: 1.0000
  Productos                 — media: 0.0000, std: 0.0000
  Ingresos_Anuales          — media: 0.0000, std: 1.0000


## 4. Comparativa de metodos de enlace y coeficiente cofenetico

El **coeficiente cofenetico** mide cuan fielmente el dendrograma preserva las distancias originales entre puntos. Valores cercanos a 1 indican mejor representacion.

In [ ]:
metodos = ["ward", "complete", "average", "single"]
dist_original = pdist(X_scaled)

resultados = []

set_link_color_palette(CLUSTER_PALETTE)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.ravel()

for i, metodo in enumerate(metodos):
    Z = linkage(X_scaled, method=metodo, metric="euclidean")
    c_coph, coph_dist = cophenet(Z, dist_original)
    resultados.append({"Metodo": metodo.capitalize(), "Cofenetico": round(c_coph, 4)})
    
    dendrogram(Z, ax=axes[i], truncate_mode="lastp", p=30, 
               leaf_rotation=90, leaf_font_size=8, color_threshold=0, above_threshold_color=INK)
    axes[i].set_title(f"{metodo.capitalize()} — cofenetico: {c_coph:.4f}", fontsize=12, fontweight="bold")
    axes[i].set_xlabel("Clientes")
    axes[i].set_ylabel("Distancia")

plt.suptitle("Comparativa de metodos de enlace", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print("\nCoeficientes cofeneticos:")
display(pd.DataFrame(resultados))

## 5. Seleccion del numero de clusters con silhouette score

Evaluamos k=2 a k=7 con el metodo Ward (mejor cofenetico en datos esfericos).

In [ ]:
Z_ward = linkage(X_scaled, method="ward", metric="euclidean")

scores = []
for k in range(2, 8):
    labels = fcluster(Z_ward, t=k, criterion="maxclust")
    sil = silhouette_score(X_scaled, labels)
    scores.append({"k": k, "silhouette": round(sil, 4)})

df_scores = pd.DataFrame(scores)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(df_scores["k"], df_scores["silhouette"], marker="o", linewidth=2, color=PURPLE)
ax.set_xlabel("Numero de clusters (k)")
ax.set_ylabel("Silhouette Score")
ax.set_title("Silhouette Score por numero de clusters")
ax.set_xticks(range(2, 8))
best_k = df_scores.loc[df_scores["silhouette"].idxmax(), "k"]
ax.axvline(x=best_k, color=INK, linestyle="--", alpha=0.6, label=f"Mejor k={best_k}")
ax.legend()
plt.tight_layout()
plt.show()

print(f"\nMejor silhouette: k={best_k} con score={df_scores['silhouette'].max():.4f}")
display(df_scores)

## 6. Silhouette plot detallado

El silhouette plot muestra la cohesion de cada punto individual dentro de su cluster. Puntos con valores negativos estan potencialmente mal asignados.

In [ ]:
labels_final = fcluster(Z_ward, t=int(best_k), criterion="maxclust")
df["Cluster"] = labels_final

sil_vals = silhouette_samples(X_scaled, labels_final)
sil_avg = silhouette_score(X_scaled, labels_final)

fig, ax = plt.subplots(figsize=(8, 6))
y_lower = 10
for cl in sorted(df["Cluster"].unique()):
    cl_sil = sil_vals[labels_final == cl]
    cl_sil.sort()
    y_upper = y_lower + len(cl_sil)
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, cl_sil, alpha=0.7, label=f"Cluster {cl}",
                      color=CLUSTER_PALETTE[(cl - 1) % len(CLUSTER_PALETTE)])
    ax.text(-0.05, y_lower + 0.5 * len(cl_sil), str(cl), fontweight="bold")
    y_lower = y_upper + 10

ax.axvline(x=sil_avg, color=INK, linestyle="--", label=f"Media: {sil_avg:.3f}")
ax.set_xlabel("Silhouette coefficient")
ax.set_ylabel("Clientes (ordenados por cluster)")
ax.set_title("Silhouette plot por cluster")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

mal_asignados = (sil_vals < 0).sum()
print(f"Clientes con silhouette negativo (posible mala asignacion): {mal_asignados} ({mal_asignados/len(df)*100:.1f}%)")

## 7. Perfil de segmentos

In [ ]:
perfil = df.groupby("Cluster")[columnas].mean().round(0)

print("Perfil medio de cada segmento:")
display(perfil)

# Heatmap normalizado para comparar
perfil_norm = (perfil - perfil.min()) / (perfil.max() - perfil.min())

fig, ax = plt.subplots(figsize=(10, 5))
ax_perfil = sns.heatmap(perfil_norm.T, annot=perfil.T.values, fmt=".0f", cmap=SEQUENTIAL_GREEN,
                         xticklabels=[f"Cluster {i}" for i in perfil.index],
                         yticklabels=columnas, ax=ax, linewidths=2, linecolor=BACKGROUND)
color_annotations(ax_perfil, perfil_norm.T.values, threshold=0.6)
ax.set_title("Perfil de segmentos (valores medios, color normalizado)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 8. Conclusiones y recomendaciones de negocio

**Hallazgos tecnicos:**
- El metodo Ward produce los clusters mas compactos y equilibrados para este dataset.
- El coeficiente cofenetico permite comparar objetivamente metodos de enlace.
- El silhouette plot revela la cohesion interna de cada cluster y detecta puntos frontera.

**Segmentos identificados:**
Los clusters reflejan niveles de vinculacion bancaria: desde clientes basicos (bajo saldo, pocos productos) hasta patrimoniales (alto saldo, muchos productos, alta antiguedad).

**Recomendaciones:**
- Asignar gestores especializados por segmento.
- Disenar productos financieros adaptados al perfil de cada cluster.
- Los clientes frontera (silhouette bajo) son candidatos a migracion al segmento superior con campanas de activacion.

**Limitaciones:**
- Dataset sintetico — la separacion real entre segmentos bancarios es mas difusa.
- No se ha incorporado comportamiento temporal (series de transacciones).
- Con mas de 5 variables, conviene aplicar PCA antes del clustering para reducir ruido.